# ⚡ New Energy Industry Agent
**Colab Free T4 GPU · Runtime → Run all**

| Component | Version | Notes |
|-----------|---------|-------|
| GPU | T4 16GB | Colab Free |
| CUDA | 12.1 | Pre-installed |
| Python | 3.12 | Pre-installed |
| NumPy | 1.26.4 | Pre-installed (do NOT change) |
| PyTorch | 2.3.0+cu121 | Pre-installed |
| Model | Qwen2.5-3B-AWQ | transformers + autoawq |

### How to use
1. (Optional) Colab sidebar 🔑 Secrets: add `HF_TOKEN` and `NGROK_TOKEN`
2. **Runtime → Run all**
3. Allow Drive access when prompted
4. First run ~5-8 min, subsequent runs ~2 min

💾 **Reconnect**: Run all again, model + data persist on Drive


In [ ]:
# Cell 1: Read Colab Secrets
import os
try:
    from google.colab import userdata
    for name in ['HF_TOKEN', 'NGROK_TOKEN']:
        try:
            val = userdata.get(name)
            if val:
                os.environ[name] = val
                print(f'OK {name}')
            else:
                print(f'Skip {name} (optional)')
        except Exception:
            print(f'Skip {name} (optional)')
except ImportError:
    print('Not in Colab')
print('Done')


In [ ]:
# Cell 2: Mount Google Drive
import os, shutil
mp = '/content/drive'
if os.path.isdir(mp) and os.listdir(mp):
    if os.path.isdir(os.path.join(mp, 'MyDrive')):
        print('Drive already mounted')
    else:
        for item in os.listdir(mp):
            p = os.path.join(mp, item)
            try:
                (shutil.rmtree if os.path.isdir(p) else os.remove)(p)
            except: pass
        from google.colab import drive; drive.mount(mp)
else:
    from google.colab import drive; drive.mount(mp)

DIRS = {
    'hf': '/content/drive/MyDrive/hf_cache',
    'vllm_kernel': '/content/drive/MyDrive/vllm_cache',
    'data': '/content/drive/MyDrive/new-energy-data',
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

os.environ['HF_HOME'] = DIRS['hf']
os.environ['HF_HUB_CACHE'] = DIRS['hf']
os.environ['VLLM_CACHE_DIR'] = DIRS['vllm_kernel']
os.environ['NEW_ENERGY_DATA_DIR'] = DIRS['data']

# Check existing cache
import sqlite3
db = os.path.join(DIRS['data'], 'electricity_cache.db')
if os.path.exists(db):
    n = sqlite3.connect(db).execute('SELECT COUNT(*) FROM electricity_prices').fetchone()[0]
    print(f'Electricity cache: {n} records (from previous session)')
else:
    print('Electricity cache: empty (first run)')
print('Drive ready')


In [ ]:
# Cell 3: Install dependencies (autoawq + transformers + gradio)
import subprocess, sys
print('Installing packages (~2 GB, please wait)...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'autoawq', 'accelerate', 'transformers',
    'gradio', 'plotly', 'pandas', 'duckduckgo_search',
    'pyngrok', 'huggingface_hub', 'httpx', 'requests'
], check=False)
print('Done')


In [ ]:
# Cell 4: Clone repo
import os
rd = '/content/new-energy-agent'
if os.path.isdir(rd):
    %cd {rd}
    !git pull -q
else:
    !git clone -q https://github.com/pai-pixel/new-energy-agent.git {rd}
    %cd {rd}
print(f'Repo: {os.getcwd()}')


In [ ]:
# Cell 5: Download model to Drive (first time ~5 min, cached ~10 sec)
import os, glob, time
from huggingface_hub import snapshot_download

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct-AWQ'
CACHE = '/content/drive/MyDrive/hf_cache'
os.makedirs(CACHE, exist_ok=True)

# Check cache
hub = os.path.join(CACHE, 'hub')
found = None
if os.path.isdir(hub):
    dirs = glob.glob(os.path.join(hub, 'models--Qwen*', 'snapshots', '*'))
    for d in dirs:
        if os.path.isdir(d) and os.listdir(d):
            found = d
            break

if found:
    gb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(found) for f in fs) / 1e9
    print(f'Model cached: {found} ({gb:.1f} GB)')
else:
    print(f'Downloading {MODEL_ID} (~2.5 GB, first time only)...')
    t0 = time.time()
    found = snapshot_download(MODEL_ID, cache_dir=CACHE, resume_download=True, max_workers=4)
    print(f'Done ({time.time()-t0:.0f}s)')

os.environ['MODEL_PATH'] = found
print(f'Model path: {found}')
print('Model on Drive, won\'t be lost on disconnect')


In [ ]:
# Cell 6: Check GPU + Load model into VRAM
import sys, os
sys.path.insert(0, '/content/new-energy-agent')

# 1. GPU check
gpu = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
if not gpu:
    print('No GPU! Runtime -> Change runtime type -> T4 GPU')
    raise SystemExit(1)
print(f'GPU: {gpu[0]}')

# 2. Load model
from src.model_engine import load_model
model_path = os.environ.get('MODEL_PATH')
print(f'Loading model: {model_path}')
print('First load compiles AWQ kernels (~30 sec), subsequent loads are instant')
model, tokenizer = load_model(model_path)
print('Model loaded!')


In [ ]:
# Cell 7: Start Agent
import os, sys, time
import gradio as gr
%cd /content/new-energy-agent
sys.path.insert(0, '/content/new-energy-agent')

from src.agent import NewEnergyAgent, create_ui, start_ngrok
from IPython.display import display, Javascript
import logging
logging.basicConfig(level=logging.WARNING)

ngrok_url = start_ngrok(7860)
agent = NewEnergyAgent()
demo = create_ui(agent)

print('=' * 50)
print('  New Energy Agent Ready!')
print('=' * 50)
if ngrok_url:
    print(f'  ngrok: {ngrok_url}')
    display(Javascript(f'window.open("{ngrok_url}", "_blank");'))
print('  Gradio URL: see cell output below for .gradio.live')
print('  Cell will keep running (normal)')
print('=' * 50)

demo.queue(max_size=32).launch(
    server_name='0.0.0.0', server_port=7860,
    share=True, show_error=True,
    css='.gradio-container{max-width:900px!important}',
    theme=gr.themes.Soft(primary_hue='green'),
)


In [ ]:
# Cell 8 (optional): Keep-alive
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
  console.log('Keepalive: '+new Date());
  document.querySelector('colab-connect-button').click();
}
setInterval(ClickConnect,60000);
'''))

import os, sqlite3
db = os.path.join(os.environ.get('NEW_ENERGY_DATA_DIR','/content/drive/MyDrive/new-energy-data'), 'electricity_cache.db')
if os.path.exists(db):
    c = sqlite3.connect(db)
    t = c.execute('SELECT COUNT(*) FROM electricity_prices').fetchone()[0]
    ps = c.execute('SELECT DISTINCT province FROM electricity_prices').fetchall()
    c.close()
    print(f'Electricity cache: {t} records | provinces: {", ".join(p[0] for p in ps)}')
print('Keep-alive started. Model + cache on Drive, survives disconnect.')


---
### Usage
- Shanghai feed-in tariff / Jiangsu desulfurized coal price
- Beijing weather / Solar subsidy policy
- Multi-turn context: 'Shanghai feed-in' -> 'Jiangsu too' -> 'commercial instead'

[GitHub](https://github.com/pai-pixel/new-energy-agent)